# 04 – Daft

> Runs against the catalog from `CATALOG` (Lakekeeper, Polaris, RustFS or Unity; default Lakekeeper). What a catalog cannot do is shown by the cell instead of aborting.

Daft (distributed DataFrames in Rust) reads Iceberg tables through a PyIceberg table
but uses its own IO. Vended credentials (STS) are taken over automatically.

> Limitation: no remote signing; writing v3 is not possible (goes through PyIceberg).

In [ ]:
from iceberg_lab.config import PROFILE, REMOTE_SIGNING, SETTINGS, WAREHOUSE_SIGNING, WAREHOUSE_STS, WAREHOUSES
from iceberg_lab.notebook import BUCKET, TRINO_SIGNING, TRINO_STS, catalog_limit, drop_table, header

CATALOG_URI = SETTINGS.catalog_uri
S3_ENDPOINT = SETTINGS.s3_endpoint
header()

In [ ]:
import daft
from daft import col
from iceberg_lab.engines import pyiceberg_catalog

catalog = pyiceberg_catalog(WAREHOUSE_STS)
catalog.create_namespace_if_not_exists("nb_daft")
if catalog.table_exists("nb_daft.products"):
    drop_table(catalog, "nb_daft.products")

In [ ]:
df = daft.from_pydict({
    "sku": [f"SKU-{i:04d}" for i in range(500)],
    "category": [["food", "tools", "toys"][i % 3] for i in range(500)],
    "price": [round(1 + i * 0.13, 2) for i in range(500)],
    "in_stock": [i % 4 != 0 for i in range(500)],
})
table = catalog.create_table("nb_daft.products", schema=df.to_arrow().schema)
df.write_iceberg(table, mode="append").show()

## Read, filter, aggregate

In [ ]:
products = daft.read_iceberg(table)
(products
    .where(col("in_stock"))
    .groupby("category")
    .agg(col("price").mean().alias("avg_price"), col("sku").count().alias("n"))
    .sort("category")
    .show())

In [ ]:
products.where(col("price") > 60).explain(show_all=True)

## Overwrite

In [ ]:
daft.read_iceberg(table).where(col("category") == "toys").write_iceberg(table, mode="overwrite").show()
daft.read_iceberg(table.refresh()).count_rows()